# Zipformer 30M — Colab T4

**Model:** [hynt/Zipformer-30M-RNNT-6000h](https://huggingface.co/hynt/Zipformer-30M-RNNT-6000h)

Dùng checkpoint **ONNX FP32**, yêu cầu CUDA provider. Không âm thầm đổi qua CPU. Dành cho thử nghiệm thực tế; chưa kiểm thử trực tiếp trên Colab T4 trong môi trường tạo notebook.

## 1. Kiểm tra Colab T4
Runtime → Change runtime type → T4 GPU. Cell dưới sẽ dừng nếu không có GPU.

In [ ]:
import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Chưa có CUDA: hãy chọn T4 GPU và kết nối lại runtime"
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


## 2. Cài đặt
Sử dụng checkpoint **FP32** gốc của `hynt/Zipformer-30M-RNNT-6000h`, không dùng INT8 vì CUDA có thể fallback. `sherpa-onnx` được yêu cầu phải thực sự có CUDA provider. Nếu wheel không hỗ trợ GPU, cell kiểm tra sẽ dừng thay vì âm thầm chạy CPU.

In [ ]:
# Kiểm tra phiên bản CUDA hệ thống và cài wheel sherpa-onnx+cuda tương ứng
!pip uninstall -y sherpa-onnx
%pip install -q \
    soundfile \
    sentencepiece \
    pynvml \
    sherpa-onnx==1.10.43+cuda12.cudnn9 -f https://k2-fsa.github.io/sherpa/onnx/cuda.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.6/255.6 MB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.18.0 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 3. Tải 3 ONNX FP32 và tokens từ model gốc

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_ID = "hynt/Zipformer-30M-RNNT-6000h"

model_dir = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        allow_patterns=[
            "encoder-epoch-20-avg-10.onnx",
            "decoder-epoch-20-avg-10.onnx",
            "joiner-epoch-20-avg-10.onnx",
            "bpe.model",
        ],
    )
)

print("Model directory:", model_dir)

for name in [
    "encoder-epoch-20-avg-10.onnx",
    "decoder-epoch-20-avg-10.onnx",
    "joiner-epoch-20-avg-10.onnx",
    "bpe.model",
]:
    assert (model_dir / name).exists(), name
    print("OK:", name)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

bpe.model:   0%|          | 0.00/268k [00:00<?, ?B/s]

encoder-epoch-20-avg-10.onnx:   0%|          | 0.00/92.2M [00:00<?, ?B/s]

decoder-epoch-20-avg-10.onnx:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

joiner-epoch-20-avg-10.onnx:   0%|          | 0.00/4.10M [00:00<?, ?B/s]

Model directory: /root/.cache/huggingface/hub/models--hynt--Zipformer-30M-RNNT-6000h/snapshots/ad6f873b5f5b6ed083821253a88d81afed2505c4
OK: encoder-epoch-20-avg-10.onnx
OK: decoder-epoch-20-avg-10.onnx
OK: joiner-epoch-20-avg-10.onnx
OK: bpe.model


In [ ]:
import urllib.request
import tarfile
from pathlib import Path

url = (
    "https://github.com/k2-fsa/sherpa-onnx/releases/"
    "download/asr-models/"
    "sherpa-onnx-zipformer-vi-30M-int8-2026-02-09.tar.bz2"
)

archive = Path("/content/zipformer_tokens.tar.bz2")

if not archive.exists():
    print("Downloading official sherpa-onnx package...")
    urllib.request.urlretrieve(url, archive)

with tarfile.open(archive, "r:bz2") as tar:
    token_files = [
        member for member in tar.getmembers()
        if Path(member.name).name == "tokens.txt"
        and member.isfile()
    ]

    if len(token_files) != 1:
        raise RuntimeError("Không tìm thấy duy nhất một tokens.txt")

    source = tar.extractfile(token_files[0])
    if source is None:
        raise RuntimeError("Không đọc được tokens.txt")

    token_data = source.read()

tokens_path = Path("/content/zipformer_tokens.txt")
tokens_path.write_bytes(token_data)

print("Tokens file:", tokens_path)
print("Size:", tokens_path.stat().st_size, "bytes")

Tokens file: /content/zipformer_tokens.txt
Size: 23238 bytes


## 2. Upload audio
Cùng một file WAV/MP3 ở cả hai notebook để so sánh. Chuyển về mono PCM 16 kHz.

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, soundfile as sf
uploaded = files.upload()
assert uploaded, "Hãy upload ít nhất một file audio"
original = next(iter(uploaded))
audio_path = '/content/asr_test_16k.wav'
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',original,'-ac','1','-ar','16000','-c:a','pcm_s16le',audio_path],check=True)
signal,sr = sf.read(audio_path)
duration = len(signal)/sr
print(f'Audio: {audio_path} | {duration:.2f}s | {sr}Hz')

Saving p_5694761_156.mp3 to p_5694761_156.mp3
Audio: /content/asr_test_16k.wav | 10.35s | 16000Hz


## DeepFilterNet3 — GPU CUDA, A/B audio gốc và khử nhiễu

DeepFilterNet3 nhận **48 kHz**; ASR nhận **16 kHz**. Để tránh làm mất thông tin trước khi enhancement, notebook lưu file 48 kHz **trực tiếp từ file upload gốc**, sau đó resample lại 16 kHz. GPU được kiểm tra bắt buộc, **không fallback CPU**. Chỉ khử nhiễu **một lần** trước benchmark ASR; các phép đo ASR độc lập, không tính thời gian DeepFilterNet3. Có thêm benchmark enhancement riêng và số liệu **pipeline ước lượng** (không phải latency đo trực tiếp toàn pipeline). Giữ nguyên bản gốc để A/B.


**Cài đặt:** chạy DeepFilterNet3 trong môi trường Python 3.11 độc lập để tương thích với PyO3 0.20; Colab Python 3.13 và môi trường ASR được giữ nguyên. CUDA chạy ở tiến trình worker, kết quả benchmark enhancement không tính thời gian khởi động worker và tải model. Lần đầu cần tải PyTorch CUDA và biên dịch Rust.

Môi trường DeepFilterNet3: **Python 3.12.3** trong `/content/df3-py312`, tạo bằng `uv` tương tự `vn_id.ipynb`; giữ nguyên Python của ASR.


In [ ]:
# ============================================================
# DeepFilterNet3 CUDA - Isolated Python 3.12.3
# Compatible with Google Colab T4
# Does NOT modify the ASR Python/CUDA environment
# ============================================================

import os
import sys
import subprocess
import shutil
from pathlib import Path


# ============================================================
# 1. Helper
# ============================================================

def run_cmd(args, env=None):
    args = [str(x) for x in args]
    print("\n+", " ".join(args), flush=True)

    subprocess.run(
        args,
        check=True,
        env=env
    )


# ============================================================
# 2. Install uv and maturin
# ============================================================

run_cmd([
    sys.executable, "-m", "pip", "install", "-q",
    "uv",
    "maturin>=1.5,<2"
])


# ============================================================
# 3. Create Python 3.12.3 environment
# ============================================================

run_cmd([
    "uv", "python", "install", "3.12.3"
])

DF_ENV = Path("/content/df3-py312")
DF_PY = DF_ENV / "bin" / "python"

if not DF_PY.exists():
    run_cmd([
        "uv", "venv",
        "--seed",
        "--python", "3.12.3",
        str(DF_ENV)
    ])

run_cmd([
    DF_PY, "-c",
    "import sys; "
    "print('Python:', sys.version); "
    "assert sys.version_info[:3] == (3, 12, 3)"
])


# ============================================================
# 4. Install PyTorch + Torchaudio CUDA 12.1
# ============================================================
# FIX:
# Install BOTH torch and torchaudio.
# They must use compatible versions and CUDA builds.

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),

    "torch==2.5.1",
    "torchaudio==2.5.1",

    "--index-url",
    "https://download.pytorch.org/whl/cu121"
])


# ============================================================
# 5. Install DeepFilterNet dependencies
# ============================================================

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),

    "numpy>=1.24,<2",
    "scipy",
    "soundfile>=0.10,<0.13",
    "loguru",
    "appdirs",
    "requests",
    "sympy",
    "packaging>=23,<25",
    "poetry-core>=1"
])


# ============================================================
# 6. Verify PyTorch / Torchaudio / CUDA
# ============================================================

run_cmd([
    DF_PY, "-c",
    "import torch, torchaudio; "
    "print('PyTorch:', torch.__version__); "
    "print('Torchaudio:', torchaudio.__version__); "
    "print('CUDA available:', torch.cuda.is_available()); "
    "print('CUDA version:', torch.version.cuda); "
    "assert torch.cuda.is_available(), 'CUDA is unavailable'"
])


# ============================================================
# 7. Install Rust
# ============================================================

rust_bin = Path.home() / ".cargo" / "bin"

build_env = os.environ.copy()

build_env["PATH"] = (
    str(rust_bin)
    + os.pathsep
    + build_env.get("PATH", "")
)

if shutil.which("cargo", path=build_env["PATH"]) is None:
    run_cmd([
        "bash", "-lc",
        'curl --proto "=https" --tlsv1.2 -sSf '
        'https://sh.rustup.rs | sh -s -- -y --profile minimal'
    ])

if (rust_bin / "rustup").exists():
    run_cmd([
        rust_bin / "rustup",
        "update",
        "stable"
    ], env=build_env)

run_cmd([
    "cargo", "--version"
], env=build_env)


# ============================================================
# 8. Clone DeepFilterNet source
# ============================================================

repo = Path("/content/DeepFilterNet-src")

if not (repo / "pyDF" / "Cargo.toml").is_file():
    run_cmd([
        "git", "clone",
        "--depth", "1",
        "https://github.com/Rikorose/DeepFilterNet.git",
        str(repo)
    ])


# ============================================================
# 9. Build deepfilterlib for Python 3.12
# ============================================================

wheels = Path("/content/df3-wheels-py312")
wheels.mkdir(parents=True, exist_ok=True)

# Reuse an existing wheel when available.
libdf_wheels = sorted(
    wheels.glob("*deepfilterlib*cp312*.whl")
)

if not libdf_wheels:

    run_cmd([
        sys.executable,
        "-m", "maturin",
        "build",
        "--release",
        "--interpreter", str(DF_PY),
        "--manifest-path", str(repo / "pyDF" / "Cargo.toml"),
        "--out", str(wheels)
    ], env=build_env)

    libdf_wheels = sorted(
        wheels.glob("*deepfilterlib*cp312*.whl")
    )

if not libdf_wheels:
    libdf_wheels = sorted(
        wheels.glob("*deepfilterlib*.whl")
    )

assert libdf_wheels, (
    f"No deepfilterlib wheel found: "
    f"{list(wheels.glob('*.whl'))}"
)


# ============================================================
# 10. Install deepfilterlib
# ============================================================

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),
    "--no-deps",
    str(libdf_wheels[-1])
])


# ============================================================
# 11. Install DeepFilterNet Python package
# ============================================================

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),
    "--no-deps",
    str(repo / "DeepFilterNet")
])


# ============================================================
# 12. Final verification
# ============================================================

verify_code = """
import sys
import torch
import torchaudio
import df
import libdf

from df.enhance import init_df, enhance

print("=" * 55)
print("DeepFilterNet3 Environment Verification")
print("=" * 55)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "DeepFilterNet3 requires CUDA"
)

print("GPU:", torch.cuda.get_device_name(0))
print("DeepFilterNet import: OK")
print("deepfilterlib import: OK")

print("=" * 55)
print("DeepFilterNet3 installation successful!")
print("=" * 55)
"""

run_cmd([
    DF_PY,
    "-c",
    verify_code
])

print("\nDeepFilterNet3 isolated Python 3.12.3 installation complete.")


+ /usr/bin/python3 -m pip install -q uv maturin>=1.5,<2

+ uv python install 3.12.3

+ uv venv --seed --python 3.12.3 /content/df3-py312

+ /content/df3-py312/bin/python -c import sys; print('Python:', sys.version); assert sys.version_info[:3] == (3, 12, 3)

+ uv pip install --python /content/df3-py312/bin/python torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

+ uv pip install --python /content/df3-py312/bin/python numpy>=1.24,<2 scipy soundfile>=0.10,<0.13 loguru appdirs requests sympy packaging>=23,<25 poetry-core>=1

+ /content/df3-py312/bin/python -c import torch, torchaudio; print('PyTorch:', torch.__version__); print('Torchaudio:', torchaudio.__version__); print('CUDA available:', torch.cuda.is_available()); print('CUDA version:', torch.version.cuda); assert torch.cuda.is_available(), 'CUDA is unavailable'

+ bash -lc curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal

+ /root/.cargo/bin/rustup update stab

In [ ]:
DF3_WORKER_SOURCE = "# Standalone DeepFilterNet3 worker: runs CUDA inference in Python 3.12.3.\n# This process is separate from the main ASR Python kernel.\nimport argparse, json, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport soundfile as sf\nfrom scipy.signal import resample_poly\nfrom df.enhance import init_df, enhance\n\np = argparse.ArgumentParser()\np.add_argument('--input', required=True)\np.add_argument('--output', required=True)\np.add_argument('--metrics', required=True)\na = p.parse_args()\nimport sys\nassert sys.version_info[:3] == (3,12,3), sys.version\nassert torch.cuda.is_available(), 'DF3 worker CUDA unavailable'\nload_start=time.perf_counter()\nmodel,state,_,_=init_df(model_base_dir='DeepFilterNet3', log_file=None)\nassert next(model.parameters()).device.type == 'cuda', 'DeepFilterNet3 is not running on CUDA'\nassert state.sr() == 48000\nload_seconds=time.perf_counter()-load_start\nwave,sr=sf.read(a.input,dtype='float32',always_2d=False)\nassert sr==48000 and wave.ndim==1, (sr,wave.shape)\nwave=torch.from_numpy(np.ascontiguousarray(wave)).unsqueeze(0)\n@torch.no_grad()\ndef run_once():\n    torch.cuda.synchronize()\n    start=time.perf_counter()\n    out=enhance(model,state,wave,pad=True)\n    torch.cuda.synchronize()\n    return out,time.perf_counter()-start\nfor _ in range(2):run_once()\ntimes=[]\nfor _ in range(5):\n    cleaned,elapsed=run_once()\n    times.append(elapsed)\nstart_rs=time.perf_counter()\ncleaned_16k=resample_poly(cleaned.squeeze(0).numpy(),up=1,down=3).astype(np.float32)\nrs_seconds=time.perf_counter()-start_rs\nsf.write(a.output,cleaned_16k,16000,subtype='PCM_16')\naudio_seconds=wave.shape[-1]/48000\nmetrics={\n 'model':'DeepFilterNet3','device':'cuda','source_sr':48000,'asr_sr':16000,\n 'worker_python':'3.12.3','model_load_seconds':load_seconds,\n 'median_enhancement_seconds':float(np.median(times)),\n 'p95_enhancement_seconds':float(np.percentile(times,95)),\n 'single_resample_seconds':rs_seconds,\n 'audio_seconds':audio_seconds,\n 'enhancement_median_rtf':float(np.median(times))/audio_seconds,\n 'notes':'DF3 warm inference only: includes Python/CUDA transfer, CPU STFT/synthesis; excludes process startup, model load, file I/O, and 48k->16k resampling. Resampling separately timed once.'}\nPath(a.metrics).write_text(json.dumps(metrics,indent=2))\nprint('DeepFilterNet3 device:',next(model.parameters()).device)\nprint('Enhancement median:',round(metrics['median_enhancement_seconds'],4),'s; RTF:',round(metrics['enhancement_median_rtf'],4))"

# DeepFilterNet3: preserve original 48 kHz audio, run separate Python 3.12.3 CUDA worker.
import json, subprocess
from pathlib import Path
raw_48k = '/content/asr_test_raw_48k.wav'
df_16k = '/content/asr_test_df3_16k.wav'
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(original),'-ac','1','-ar','48000',
                '-c:a','pcm_f32le',raw_48k],check=True)
worker_path=Path('/content/df3_worker.py')
worker_path.write_text(DF3_WORKER_SOURCE)
subprocess.run([str(DF_PY),str(worker_path),'--input',raw_48k,'--output',df_16k,
                '--metrics','/content/df3_metrics.json'],check=True)
DF3_METRICS=json.loads(Path('/content/df3_metrics.json').read_text())
assert Path(df_16k).exists() and Path(df_16k).stat().st_size>44
print('48k input:',raw_48k,'| 16k clean:',df_16k)
print('Enhancement median:',DF3_METRICS['median_enhancement_seconds'],
      's, RTF:',DF3_METRICS['enhancement_median_rtf'])


48k input: /content/asr_test_raw_48k.wav | 16k clean: /content/asr_test_df3_16k.wav
Enhancement median: 0.06674375000000055 s, RTF: 0.006451801296522721


## 4. Kiểm tra CUDA ở ONNX Runtime và Sherpa
Nếu gặp lỗi CUDA provider không khả dụng, runtime `sherpa-onnx` của phiên bản pip chưa hỗ trợ CUDA. **Không đổi thành CPU rồi báo là GPU.**

In [ ]:
import onnxruntime as ort
print('ONNX Runtime providers:',ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'onnxruntime-gpu chưa nhận CUDA; xem compatibility CUDA/cuDNN'
import sherpa_onnx
print('sherpa-onnx loaded')

ONNX Runtime providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
sherpa-onnx loaded


## 5. Tạo recognizer GPU (chỉ tạo một lần)
Giữ recognizer trong notebook để benchmark **warm inference**, không chạy subprocess cho mỗi audio.

In [ ]:
import time
import sherpa_onnx
start_load=time.perf_counter()
recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
    tokens=str(tokens_path),
    encoder=str(model_dir/'encoder-epoch-20-avg-10.onnx'),
    decoder=str(model_dir/'decoder-epoch-20-avg-10.onnx'),
    joiner=str(model_dir/'joiner-epoch-20-avg-10.onnx'),
    num_threads=1,
    sample_rate=16000,
    feature_dim=80,
    decoding_method='greedy_search',
    provider='cuda',
)
load_time=time.perf_counter()-start_load
print('Recognizer initialized with provider=cuda; load time:',round(load_time,3),'s')

Recognizer initialized with provider=cuda; load time: 1.481 s


## 6. Benchmark chuẩn hóa: 5 warm-up + 20 lần đo

Giữ recognizer trong bộ nhớ. Thời gian đo: tạo stream + nạp waveform + decode, không bao gồm load model, đọc file hoặc network. Theo dõi **GPU memory used** bằng `nvidia-smi` trong lúc chạy; giá trị này là tổng bộ nhớ GPU sử dụng, không phải VRAM riêng của model. RTF tính bằng median latency / audio duration. Decoder: greedy search. Kết quả CSV và JSON lưu tại `/content`.

In [ ]:
import numpy as np, soundfile as sf, time, statistics, json, csv, subprocess, threading
from pathlib import Path

for variant, audio_path in [('raw', '/content/asr_test_16k.wav'), ('df3', '/content/asr_test_df3_16k.wav')]:
    print(f"\n=== Zipformer / {variant} ===")
    signal, sr = sf.read(audio_path, dtype='float32')
    assert sr == 16000 and signal.ndim == 1
    audio_seconds = len(signal) / sr
    N_WARMUP, N_RUNS = 5, 20

    def gpu_used_mib():
        p = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                           capture_output=True, text=True, check=True)
        return int(p.stdout.strip().splitlines()[0].strip())

    class GPUMonitor:
        def __init__(self):
            self.samples = []
            self.stop = threading.Event()
        def __enter__(self):
            self.thread = threading.Thread(target=self._poll, daemon=True)
            self.thread.start()
            return self
        def _poll(self):
            while not self.stop.is_set():
                try: self.samples.append(gpu_used_mib())
                except Exception: pass
                self.stop.wait(0.05)
        def __exit__(self, *args):
            self.stop.set()
            self.thread.join(timeout=3)

    def recognize_once():
        start = time.perf_counter()
        stream = recognizer.create_stream()
        stream.accept_waveform(sr, signal)
        recognizer.decode_stream(stream)
        elapsed = time.perf_counter() - start
        return stream.result.text, elapsed

    for _ in range(N_WARMUP):
        recognize_once()

    baseline_mib = gpu_used_mib()
    with GPUMonitor() as monitor:
        results = [recognize_once() for _ in range(N_RUNS)]

    latencies = [elapsed for _, elapsed in results]
    transcripts = [text for text, _ in results]
    assert len(set(transcripts)) == 1, 'Transcript thay đổi giữa các lần chạy!'
    median_s = float(np.median(latencies))
    p95_s = float(np.percentile(latencies, 95))
    peak_mib = max(monitor.samples) if monitor.samples else None
    report = {
        'model': 'hynt/Zipformer-30M-RNNT-6000h',
        'audio_variant':variant,
        'runtime': 'sherpa-onnx ONNX FP32',
        'gpu': 'Tesla T4', 'requested_provider': 'cuda',
        'decoding_method': 'greedy_search',
        'warmup_runs': N_WARMUP, 'measured_runs': N_RUNS,
        'audio_seconds': audio_seconds,
        'median_latency_seconds': median_s,
        'p95_latency_seconds': p95_s,
        'median_rtf': median_s / audio_seconds,
        'p95_rtf': p95_s / audio_seconds,
        'gpu_memory_baseline_mib': baseline_mib,
        'gpu_memory_peak_total_mib': peak_mib,
        'text': transcripts[0],
        'deepfilternet3': DF3_METRICS if variant == 'df3' else None,
        'approx_pipeline_median_seconds': median_s + DF3_METRICS['median_enhancement_seconds'] + DF3_METRICS['single_resample_seconds'] if variant == 'df3' else None,
        'notes': 'nvidia-smi GPU total memory sampled; not isolated model VRAM. Inference timing excludes audio file I/O and model load.'
    }
    rows = [{'run': i+1, 'latency_seconds': round(t, 6), 'rtf': round(t/audio_seconds, 6)}
            for i, t in enumerate(latencies)]
    with open(f'/content/zipformer30m_{variant}_benchmark.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['run', 'latency_seconds', 'rtf'])
        writer.writeheader(); writer.writerows(rows)
    Path(f'/content/zipformer30m_{variant}_benchmark.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
    print('Transcript:', transcripts[0])
    print(f'Median: {median_s:.4f}s | P95: {p95_s:.4f}s | Median RTF: {median_s/audio_seconds:.4f}')
    print(f'GPU total memory used baseline/peak: {baseline_mib}/{peak_mib} MiB')
    print('Saved variant:',variant)



=== Zipformer / raw ===
Transcript: VĂN BẢN NÀY CÓ THỂ ĐƯỢC THAY THẾ BẰNG BẤT KỲ VĂN BẢN NÀO TRONG HỘP VĂN BẢN NÀY BẠN CÓ THỂ NHẬP HOẶC DÁN VĂN BẢN CỦA RIÊNG MÌNH BẠN CÓ THỂ SỬ DỤNG NHIỀU GIỌNG NÓI CHO MỘT VĂN BẢN
Median: 0.4283s | P95: 0.9710s | Median RTF: 0.0414
GPU total memory used baseline/peak: 3/3 MiB
Saved variant: raw

=== Zipformer / df3 ===
Transcript: VĂN BẢN NÀY CÓ THỂ ĐƯỢC THAY THẾ BẰNG BẤT KỲ VĂN BẢN NÀO TRONG HỘP VĂN BẢN NÀY BẠN CÓ THỂ NHẬP HOẶC DÁN VĂN BẢN CỦA RIÊNG MÌNH BẠN CÓ THỂ SỬ DỤNG NHIỀU GIỌNG NÓI CHO MỘT VĂN BẢN
Median: 0.4330s | P95: 0.4655s | Median RTF: 0.0419
GPU total memory used baseline/peak: 3/3 MiB
Saved variant: df3


**Lưu ý:** `nvidia-smi` lấy mẫu 50 ms nên có thể bỏ sót đỉnh VRAM rất ngắn; kết quả memory là tổng GPU sử dụng. Notebook này dùng greedy search, khớp chế độ mặc định của bản Zipformer notebook gốc. File benchmark **không phải WER**; muốn đánh giá độ chính xác cần transcript chuẩn.

In [ ]:
from google.colab import files
for variant in ('raw','df3'):
    for ext in ('csv','json'):
        files.download(f'/content/zipformer30m_{variant}_benchmark.{ext}')
files.download('/content/df3_metrics.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>